In [3]:
# Fine-tuning ResNet model

# Loading CNN model
from torchvision import models
import torch

restnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
restnet.fc = torch.nn.Linear(restnet.fc.in_features, 10)

In [4]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as T

# Preprocessing data
train_transforms = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

test_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

train_data = CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transforms
)

test_data = CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transforms
)

train_loader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False,
    num_workers=2
) 

In [7]:
# Training Loop + Validation 
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(restnet.parameters(), lr=2e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
restnet.to(device)

# Evaluation Function
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy

# Training Loop
for epoch in range(10): # Number of epochs
    restnet.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = restnet(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, accuracy = evaluate(restnet, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/10], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch [1/10], Train Loss: 0.4147, Val Loss: 0.4654, Val Accuracy: 84.10%
Epoch [2/10], Train Loss: 0.4037, Val Loss: 0.4630, Val Accuracy: 84.62%
Epoch [3/10], Train Loss: 0.3833, Val Loss: 0.4478, Val Accuracy: 84.68%
Epoch [4/10], Train Loss: 0.3736, Val Loss: 0.4510, Val Accuracy: 84.94%
Epoch [5/10], Train Loss: 0.3623, Val Loss: 0.4519, Val Accuracy: 84.99%
Epoch [6/10], Train Loss: 0.3454, Val Loss: 0.4669, Val Accuracy: 84.97%
Epoch [7/10], Train Loss: 0.3335, Val Loss: 0.4439, Val Accuracy: 85.29%
Epoch [8/10], Train Loss: 0.3236, Val Loss: 0.4342, Val Accuracy: 86.09%
Epoch [9/10], Train Loss: 0.3182, Val Loss: 0.4394, Val Accuracy: 85.78%
Epoch [10/10], Train Loss: 0.3058, Val Loss: 0.4348, Val Accuracy: 85.86%
